# 무릎 MRI 12개 소견 분류 — 단순 CNN 기준선 (교육용) · v1.1.0 (증강 비교판)

> 버전 v1.1.0 (2026-10-07) — v1.0.1 대비: 증강 옵션(KneeSet), 학습 함수(run_experiment), 정답지 4개 짝 비교(compare), 실험 계획 셀. 이전 이력: v1.0.1 — v1.0.0 대비: 데이터 셀이 Drive 전체에서 images.npy를 찾고 파일 크기를 확인합니다.

RSNA Knee 대회 데이터를 **줄인 학습용 세트**로 단순 2.5D CNN을 처음부터 끝까지 학습해 보는 노트북입니다.

> 규칙: 대회 데이터는 팀 밖과 비공개로 공유하면 안 됩니다. 이 데이터와 노트북은 본인 계정에서만 쓰세요.

---

## ⏱ 걸리는 시간 (Colab 무료 T4 기준, 추정치)

| 단계 | 시간 |
|---|---|
| 0. 런타임 준비, 패키지 설치 | 1~2분 |
| 1. 데이터 받기 (2.6GB) + 로컬 디스크로 복사 | 2~5분 |
| 2. 데이터 살펴보기 | 1분 이내 |
| 5. 학습 — ResNet34, fold 1개, 8 epoch | **약 20~30분** (epoch당 2~3분) |
| 6. 평가 | 1분 이내 |
| (선택) 5개 fold 모두 | 약 2~2.5시간 |
| (선택) ResNet50으로 바꾸면 | 약 1.5~2배 |

무료 Colab은 오래 쓰지 않으면 세션이 끊길 수 있습니다. **fold 하나씩** 돌리고 결과를 저장하세요.

---

## 🧠 원리 — 이 노트북이 하는 일

**1) 문제:** 무릎 MRI 검사 하나를 보고 12개 소견(ACL 파열, 반월상연골 파열, 관절염, 삼출 등)이 **각각 있는지 확률**을 냅니다. 소견이 여러 개 동시에 있을 수 있어서 *다중 라벨 분류*입니다.

**2) 입력 — "2.5D":** MRI는 3D지만, 여기서는 검사마다 **2D 슬라이스 12장**을 씁니다. 원본 DICOM을 무릎 중심 140mm로 자르고, 실제 위치 순서대로 정렬하고, 밝기를 8비트로 맞춘 캐시에서 골라낸 장들입니다. 3D 모델보다 가볍고 빠르면서 여러 단면을 볼 수 있습니다.

**3) 전이학습:** ImageNet(일반 사진 128만 장)으로 미리 학습된 ResNet을 가져와 시작합니다. 테두리·질감 같은 기본 시각 특징을 이미 알고 있어서, 적은 의료영상으로도 빨리 배웁니다.

**4) 슬라이스 평균 (MIL pooling):** 슬라이스 12장 각각을 CNN에 넣어 특징 벡터를 얻고, **12개를 평균**내서 검사 하나의 특징으로 만듭니다. 그 위에 선형층 하나가 12개 점수를 냅니다. 어떤 장에 소견이 보이는지 몰라도 학습할 수 있는 단순한 방법입니다.

**5) 라벨과 손실:** 정답 라벨은 판독 보고서를 LLM이 읽고 만든 **확률(soft label)**입니다. 공식 정답(전문의가 영상을 다시 판독)과 다를 수 있는 **잡음 섞인 라벨**입니다. 손실은 이진 교차엔트로피(BCE)이고, 드문 소견(MCL, 골절 등)의 양성 비중을 키우는 `pos_weight`를 씁니다.

**6) 검증 — fold와 OOF:** 검사를 5묶음(fold)으로 나눠, 4개로 학습하고 나머지 1개로 평가합니다. 같은 환자의 검사는 같은 fold에 들어가게 나눠져 있어서 정보가 새지 않습니다. 5번 돌리면 모든 검사에 대해 "학습 때 못 본 예측(OOF)"이 생깁니다.

**7) 평가 지표 — AUC:** 양성 검사와 음성 검사를 하나씩 뽑았을 때 모델이 양성에 더 높은 점수를 줄 확률입니다. 1.0이 완벽, 0.5가 무작위. 12개 소견의 평균(macro AUC)이 대회 점수 방식입니다.
- 주의: **채점 정답지(라벨 세트)가 바뀌면 점수의 크기도 바뀝니다.** 같은 정답지 안에서만 모델끼리 비교하세요.

**8) 혼합 정밀도(AMP):** 계산 일부를 16비트로 해서 T4에서 약 2배 빠르게, 메모리는 적게 씁니다.

## 0. 런타임 설정
Colab 메뉴 → **런타임 → 런타임 유형 변경 → T4 GPU**를 고르세요.

In [ ]:
print('notebook v1.1.0')
# GPU가 잡혔는지 확인합니다. "Tesla T4, 15360 MiB" 비슷하게 나오면 정상입니다.
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
# timm: 사전학습된 이미지 모델(ResNet, EfficientNet 등)을 한 줄로 불러오는 라이브러리
# scikit-learn: AUC 같은 평가 지표 계산용
!pip -q install timm scikit-learn

## 1. 데이터 준비 (둘 중 하나)
- **A. Google Drive:** `images.npy`(약 2.6GB)와 `meta.csv`를 Drive의 `knee_compact/` 폴더에 올려 두었을 때.
- **B. 비공개 Kaggle 데이터셋:** 두 파일을 본인 계정의 비공개 데이터셋으로 올려 두었을 때. Kaggle 토큰 파일은 **본인이 직접** 업로드합니다.

어느 쪽이든 마지막에 Colab 로컬 디스크(`/content`)로 복사합니다. Drive에서 직접 읽으면 학습 중 데이터 읽기가 느려지기 때문입니다.

In [ ]:
import os, shutil

SOURCE = 'drive'                                              # 'drive' 또는 'kaggle'
KAGGLE_DATASET = 'YOUR_KAGGLE_ID/knee-compact-learning-v1'    # B 방법일 때 본인 데이터셋 이름으로 바꾸세요
DATA = '/content/knee_compact'                                # 학습 때 읽을 로컬 폴더
os.makedirs(DATA, exist_ok=True)

if SOURCE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')                             # 브라우저 팝업에서 Google 계정 권한을 허용
    import glob
    EXPECTED = 2614972544                                     # images.npy 정상 크기(바이트). 다르면 업로드가 덜 끝난 것
    # 폴더 이름이나 위치가 달라도 찾도록 MyDrive 전체에서 images.npy를 검색 (Drive가 크면 수십 초 걸릴 수 있음)
    hits = sorted(glob.glob('/content/drive/MyDrive/**/images.npy', recursive=True)[:5] +
                  glob.glob('/content/drive/MyDrive/images.npy'))
    if not hits:
        print('MyDrive 최상위 목록:', sorted(os.listdir('/content/drive/MyDrive'))[:50])
        raise FileNotFoundError('Drive에서 images.npy를 찾지 못했습니다. 업로드가 끝났는지(drive.google.com 우측 아래 진행 표시), '
                                '"공유 문서함"이 아니라 "내 드라이브"에 있는지 확인하세요. 업로드 직후라면 런타임 > 세션 다시 시작 후 다시 실행하세요.')
    src = os.path.dirname(hits[0])
    size = os.path.getsize(hits[0])
    print('찾은 위치:', src, '| images.npy 크기:', size)
    if size != EXPECTED:
        raise RuntimeError(f'images.npy 크기가 {size}바이트로 정상({EXPECTED})과 다릅니다. 업로드가 아직 진행 중이거나 중간에 끊긴 것입니다.')
    for name in ['images.npy', 'meta.csv']:                   # 로컬 디스크로 복사 (1~3분)
        shutil.copy(f'{src}/{name}', f'{DATA}/{name}')
else:
    from google.colab import files
    print('kaggle.json 또는 access_token 파일을 선택하세요')
    up = files.upload()                                       # 토큰 파일을 본인이 직접 선택
    name = list(up)[0]
    os.makedirs('/root/.kaggle', exist_ok=True)
    dst = '/root/.kaggle/kaggle.json' if name.endswith('.json') else '/root/.kaggle/access_token'
    open(dst, 'wb').write(up[name])
    os.chmod(dst, 0o600)                                      # 토큰 파일은 본인만 읽을 수 있게
    if not name.endswith('.json'):
        os.environ['KAGGLE_API_TOKEN'] = open(dst).read().strip()   # 새 형식 토큰은 환경 변수로 전달
    !pip -q install -U kaggle
    !kaggle datasets download -d {KAGGLE_DATASET} -p {DATA} --unzip -q

print(os.listdir(DATA))                                       # images.npy, meta.csv가 보이면 준비 완료

## 2. 데이터 살펴보기
`images.npy`는 **메모리 매핑(mmap)**으로 엽니다. 2.6GB를 RAM에 한꺼번에 올리지 않고, 필요한 검사만 그때그때 디스크에서 읽는 방식입니다.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt

# 12개 소견 이름 (meta.csv의 열 이름과 같은 순서)
T = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA',
     'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']

X = np.load(f'{DATA}/images.npy', mmap_mode='r')   # 모양: [검사 수, 슬라이스 12, 224, 224], 값 0~255
meta = pd.read_csv(f'{DATA}/meta.csv')               # 검사 ID, fold 번호, Dread 라벨(학습용), 비교용 라벨들
print('이미지 배열:', X.shape, X.dtype, '| 검사 수:', len(meta))
print('fold별 검사 수:', meta['fold'].value_counts().sort_index().to_dict())

# 라벨 확률이 0.5 이상인 비율 = 대략적인 양성 비율. 드문 소견일수록 학습이 어렵습니다.
print('Dread 라벨 기준 양성 비율:')
print((meta[T] >= 0.5).mean().round(3).to_string())

# 검사 하나의 슬라이스 12장을 그려 봅니다.
i = 0
fig, ax = plt.subplots(2, 6, figsize=(15, 5))
for j in range(12):
    ax[j // 6, j % 6].imshow(X[i, j], cmap='gray')
    ax[j // 6, j % 6].set_title(f'slice {j}', fontsize=8)
    ax[j // 6, j % 6].axis('off')
plt.suptitle('검사 1건의 슬라이스 12장 (앞쪽은 시상면, 뒤쪽은 관상면·축상면)')
plt.show()
print('이 검사의 라벨:', meta.loc[i, T].round(2).to_dict())

## 3. 데이터셋과 증강 옵션
- 모델 입력: 슬라이스 12장 [12, 224, 224], 값 0~1.
- 증강 종류는 `AUGS`에 있습니다: none, base(대조군), flip, affine, sdrop.
- 상위권 방법: 증강을 **하나씩** 바꿔 대조군과 비교합니다.


In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# 증강 종류. 실험할 때는 한 번에 "하나만" 바꿔서 대조군('base')과 비교합니다.
#  'none'   : 증강 없음
#  'base'   : 밝기·대비 조금 + 상하좌우 이동 (원래 노트북과 같음 → 대조군)
#  'flip'   : base + 50% 확률로 좌우 뒤집기 (왼쪽 무릎 ↔ 오른쪽 무릎을 흉내)
#  'affine' : base + 작은 회전(±10°)과 확대·축소(±10%) (자세·화면 크기 차이를 흉내)
#  'sdrop'  : base + 슬라이스 12장 중 2장을 무작위로 지우기 (일부 시퀀스가 빠진 검사를 흉내)
AUGS = ['none', 'base', 'flip', 'affine', 'sdrop']

class KneeSet(Dataset):
    """검사 하나 = (슬라이스 12장 텐서 [12, 224, 224], 12개 soft label)"""
    def __init__(self, rows, aug='none'):
        assert aug in AUGS, aug
        self.rows = list(rows)                                   # 이 데이터셋에 쓸 meta.csv 행 번호
        self.aug = aug                                           # 검증용은 항상 'none'
        self.y = meta.loc[self.rows, T].values.astype('float32') # 학습 라벨: Dread soft label (0~1)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        x = torch.from_numpy(np.array(X[self.rows[i]], dtype=np.float32) / 255.0)   # [12, H, W], 0~1
        if self.aug != 'none':
            # 대비 ±5%, 밝기 ±0.025 (장비마다 밝기가 다른 것을 흉내)
            x = x * (1 + 0.1 * (torch.rand(1) - 0.5)) + 0.05 * (torch.rand(1) - 0.5)
            # 상하좌우 최대 12픽셀 이동 (무릎 위치 차이를 흉내)
            dx, dy = np.random.randint(-12, 13, 2)
            x = torch.roll(x, shifts=(int(dy), int(dx)), dims=(1, 2))
        if self.aug == 'flip' and torch.rand(1) < 0.5:
            x = torch.flip(x, dims=(2,))                         # 가로 방향 뒤집기 (12장 모두 같이)
        if self.aug == 'affine':
            a = np.deg2rad(np.random.uniform(-10, 10)); s = np.random.uniform(0.9, 1.1)
            theta = torch.tensor([[np.cos(a) / s, -np.sin(a) / s, 0.0],
                                  [np.sin(a) / s, np.cos(a) / s, 0.0]], dtype=torch.float32)[None]
            grid = F.affine_grid(theta, (1, 1, x.shape[1], x.shape[2]), align_corners=False)
            x = F.grid_sample(x[:, None], grid.expand(x.shape[0], -1, -1, -1), align_corners=False).squeeze(1)
        if self.aug == 'sdrop':
            x[np.random.choice(x.shape[0], 2, replace=False)] = 0  # 2장을 검은 화면으로
        return x.clamp(0, 1), torch.from_numpy(self.y[i])


## 4. 모델 — 슬라이스별 CNN 특징 → 평균 → 12개 출력
상위권 참가자의 조언대로 **어텐션 없이 단순 pooling**부터 시작합니다. `BACKBONE` 한 줄만 바꾸면 다른 모델을 시험할 수 있습니다 (`resnet34`, `resnet50`, `efficientnet_b0` 등).

In [ ]:
import timm

BACKBONE = 'resnet34'   # 바꿔 보기: 'resnet50', 'efficientnet_b0', 'convnext_tiny'

# ImageNet 사전학습 모델은 이 평균/표준편차로 정규화된 입력을 기대합니다.
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

class SliceMean(nn.Module):
    def __init__(self, name):
        super().__init__()
        # num_classes=0 → 분류층을 떼고 특징 벡터만 내보내는 "인코더"로 씁니다.
        self.enc = timm.create_model(name, pretrained=True, num_classes=0)
        # 특징 벡터 → 12개 소견 점수(logit)
        self.head = nn.Linear(self.enc.num_features, 12)

    def forward(self, x):                              # x: [배치 B, 슬라이스 12, H, W]
        b, s, h, w = x.shape
        x = x.reshape(b * s, 1, h, w)                  # 슬라이스를 각각 독립된 이미지로 펼침
        x = x.repeat(1, 3, 1, 1)                       # 흑백 1채널 → 3채널 (사전학습 모델이 3채널을 기대)
        x = (x - MEAN.to(x.device)) / STD.to(x.device) # ImageNet 방식 정규화
        f = self.enc(x)                                # [B*12, 특징 차원]
        f = f.reshape(b, s, -1).mean(1)                # 슬라이스 12장의 특징을 평균 → 검사 하나의 특징
        return self.head(f)                            # [B, 12] — 시그모이드 전 점수(logit)

## 5. 학습 함수 — 실험 하나 = (이름, 증강, seed)
- 학습 설정(epoch, 배치, 학습률, 백본)은 실험 중에 **고정**합니다. 비교할 것 하나만 바꿉니다.
- 결과(검증 예측)는 Drive의 `knee_runs/`에 저장되어, 연결이 끊겨도 끝난 실험은 다시 돌리지 않습니다.
- 이 셀은 함수만 정의합니다. 실제 실행은 맨 아래 7번 셀에서 합니다.


In [ ]:
from sklearn.metrics import roc_auc_score
import os, time, random

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
EPOCHS, BS, LR = 8, 8, 3e-4                 # 학습 횟수, 배치(검사 수), 최대 학습률 — 실험 중에는 고정!
# 결과는 Drive에 저장합니다. Colab 연결이 끊겨도 이미 끝난 실험은 다시 돌리지 않고 건너뜁니다.
SAVE = '/content/drive/MyDrive/knee_runs' if os.path.isdir('/content/drive/MyDrive') else '/content/knee_runs'
os.makedirs(SAVE, exist_ok=True)
print('결과 저장 위치:', SAVE)

def seed_all(seed):
    """같은 seed면 같은 초기값·같은 데이터 순서·같은 증강이 나오도록 고정합니다."""
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def worker_init(worker_id):
    # DataLoader 일꾼마다 numpy 난수를 다르게, 그러나 재현 가능하게 (안 하면 일꾼들이 같은 증강을 반복)
    np.random.seed(torch.initial_seed() % 2**32)

def run_experiment(name, aug='base', seed=42, fold=0, backbone=None, epochs=None):
    """fold 하나를 학습하고 검증 예측(OOF)을 Drive에 저장합니다. 이미 있으면 불러오기만 합니다."""
    backbone = backbone or BACKBONE; epochs = epochs or EPOCHS
    out = f'{SAVE}/{name}_seed{seed}_fold{fold}.csv'
    if os.path.exists(out):
        print('이미 있음 → 건너뜀:', out)
        return
    seed_all(seed)
    tr = meta.index[meta['fold'] != fold]; va = meta.index[meta['fold'] == fold]
    # pos_weight: 드문 소견의 양성 오류를 더 크게 벌함 (1~10으로 자름)
    prev = np.clip((meta.loc[tr, T].values >= 0.5).mean(0), 0.03, 0.7)
    pos_weight = torch.tensor(np.clip((1 - prev) / prev, 1, 10), dtype=torch.float32, device=dev)
    g = torch.Generator(); g.manual_seed(seed)
    dl_tr = DataLoader(KneeSet(tr, aug), BS, shuffle=True, num_workers=2, drop_last=True,
                       generator=g, worker_init_fn=worker_init)
    dl_va = DataLoader(KneeSet(va, 'none'), BS, shuffle=False, num_workers=2)
    model = SliceMean(backbone).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.02)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=epochs * len(dl_tr), pct_start=0.15)
    use_amp = dev == 'cuda'
    scaler = torch.amp.GradScaler(enabled=use_amp)
    Yd = np.rint(meta.loc[va, T].values).astype(int)
    print(f'== {name} | aug={aug} seed={seed} fold={fold} {backbone} | 학습 {len(tr)} / 검증 {len(va)}')
    for ep in range(epochs):
        t0 = time.time(); model.train(); tot = 0
        for x, y in dl_tr:
            x, y = x.to(dev), y.to(dev)
            with torch.autocast(dev, dtype=torch.float16, enabled=use_amp):
                loss = F.binary_cross_entropy_with_logits(model(x).float(), y, pos_weight=pos_weight)
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 3.0)
            scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item()
        model.eval(); P = []
        with torch.no_grad(), torch.autocast(dev, dtype=torch.float16, enabled=use_amp):
            for x, _ in dl_va:
                P.append(torch.sigmoid(model(x.to(dev)).float()).cpu().numpy())
        P = np.concatenate(P)
        print(f'  epoch {ep + 1}/{epochs}: loss {tot / len(dl_tr):.4f} | val AUC (Dread) {fast_macro(Yd, P):.4f} | {time.time() - t0:.0f}s')
    # 마지막 epoch의 예측을 저장합니다 (epoch을 점수 보고 고르면 검증 점수가 부풀려지므로 고르지 않음).
    pd.DataFrame(P, columns=T, index=meta.loc[va, 'StudyInstanceUID']).to_csv(out)
    print('  저장:', out)

def fast_auc(y, p):
    """순위 기반 AUC (roc_auc_score와 같은 값, 더 빠름). 양성이나 음성이 없으면 nan."""
    n1 = y.sum(); n0 = len(y) - n1
    if n1 == 0 or n0 == 0:
        return np.nan
    r = pd.Series(p).rank().values
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

def fast_macro(Y, P, M=None):
    """12개 소견 AUC의 평균. M이 있으면 각 소견에서 M이 True인 검사만 씁니다(합의 기준)."""
    vals = []
    for t in range(Y.shape[1]):
        m = np.ones(len(Y), bool) if M is None else M[:, t]
        vals.append(fast_auc(Y[m, t], P[m, t]))
    return np.nanmean(vals)


## 6. 비교 함수 — 정답지 4개로 대조군과 짝 비교
- **흔들림**: 대조군 seed 42와 43의 점수 차이. 같은 설정에서도 이만큼은 우연히 움직입니다.
- **Δ와 구간**: 변형 − 대조군. 구간이 0을 넘고, Δ가 흔들림보다 크고, 네 정답지에서 방향이 같아야 진짜 개선으로 봅니다.
- 합의(consensus) 기준이 가장 깨끗한 자이고, 학습에 쓴 Dread 기준은 Dread에 유리합니다.


In [ ]:
import glob

def references(rows):
    """정답지 네 가지. 모두 판독문에서 뽑은 라벨이라 공식 정답은 아닙니다."""
    d = np.rint(meta.loc[rows, T].values).astype(int)
    f = meta.loc[rows, [f'flight_{t}' for t in T]].values.astype(int)
    r = meta.loc[rows, [f'relabel_{t}' for t in T]].values.astype(int)
    st = np.stack([d, f, r])
    cons = st.min(0) == st.max(0)                      # 세 라벨이 모두 같은 칸만 True
    return {'Dread': (d, None), 'flight': (f, None), 'relabel': (r, None), 'consensus': (d, cons)}

def load_run(name, seed, fold=0):
    p = pd.read_csv(f'{SAVE}/{name}_seed{seed}_fold{fold}.csv', index_col=0)
    rows = meta.set_index('StudyInstanceUID').loc[p.index, 'row'].values
    return rows, p[T].values

def compare(control, variants, fold=0, n_boot=200):
    """대조군 대비 각 변형의 점수 차이를, 정답지 4개 각각에 대해 보여줍니다.
    - 점수는 seed별 점수의 평균, 차이는 (변형 평균 − 대조군 평균).
    - 구간은 검증 검사를 복원추출해 200번 다시 계산한 95% 범위 (모든 실험에 같은 표본을 써서 짝 비교).
    - '흔들림'은 대조군 seed끼리의 차이: 이보다 작은 차이는 우연일 수 있습니다."""
    def seeds_of(name):
        return sorted(int(f.split('_seed')[1].split('_')[0]) for f in glob.glob(f'{SAVE}/{name}_seed*_fold{fold}.csv'))
    runs = {n: [load_run(n, s, fold) for s in seeds_of(n)] for n in [control] + variants}
    rows = runs[control][0][0]
    for n, rs in runs.items():
        assert rs, f'{n}: 저장된 실험이 없습니다'
        assert all((r == rows).all() for r, _ in rs), '검증 검사 순서가 다릅니다'
    refs = references(rows)
    rng = np.random.default_rng(0)
    boots = [rng.integers(0, len(rows), len(rows)) for _ in range(n_boot)]
    def mean_score(rs, Y, M, idx=None):
        idx = np.arange(len(rows)) if idx is None else idx
        return np.mean([fast_macro(Y[idx], P[idx], None if M is None else M[idx]) for _, P in rs])
    lines = []
    for ref, (Y, M) in refs.items():
        c = mean_score(runs[control], Y, M)
        cs = [fast_macro(Y, P, M) for _, P in runs[control]]
        noise = max(cs) - min(cs) if len(cs) > 1 else float('nan')
        lines.append(f'[{ref}] {control} {c:.4f} (seed {len(cs)}개, 흔들림 {noise:.4f})')
        for v in variants:
            d = mean_score(runs[v], Y, M) - c
            bd = [mean_score(runs[v], Y, M, b) - mean_score(runs[control], Y, M, b) for b in boots]
            lo, hi = np.nanpercentile(bd, [2.5, 97.5])
            lines.append(f'    {v:8s} Δ {d:+.4f}  [{lo:+.4f}, {hi:+.4f}]  (seed {len(runs[v])}개)')
    print('\n'.join(lines))
    return '\n'.join(lines)


## 7. 실험 실행 — 대조군 2번 + 바꾼 것 하나
- 처음에는 대조군(seed 42, 43)과 좌우 뒤집기(flip)만 돌립니다 (약 1.5~2시간).
- 끝나면 마지막 출력(비교표)을 복사해 대화창에 붙여 주세요. 다음에 무엇을 바꿀지 같이 정합니다.


In [ ]:
# ===== 실험 계획: 대조군 2번(seed 42, 43) → 증강 하나씩 =====
# T4 기준 실험 1개(ResNet34, fold 0, 8 epoch)에 약 20~30분. 아래 4개면 약 1.5~2시간.
# 연결이 끊기면 이 셀을 다시 실행하세요. 끝난 실험은 Drive에 있어서 건너뜁니다.
PLAN = [
    ('ctrl', 'base', 42),   # 대조군 1
    ('ctrl', 'base', 43),   # 대조군 2 → 두 대조군 차이 = 우연에 의한 흔들림 폭
    ('flip', 'flip', 42),   # 바꾼 것 1개: 좌우 뒤집기
    ('flip', 'flip', 43),
    # 다음 회차에 하나씩 추가: ('affine', 'affine', 42), ('affine', 'affine', 43), ('sdrop', 'sdrop', 42), ...
]
for name, aug, seed in PLAN:
    run_experiment(name, aug=aug, seed=seed)

# 결과 비교 — 이 출력 전체를 복사해서 알려 주세요.
done = sorted({n for n, _, _ in PLAN} - {'ctrl'})
summary = compare('ctrl', done)
